# Model Context Protocol (MCP)

We can use any LLM with MCP. In this repository we will use `llama3.2:1b` and `llama3.2:3b`

You can also download other models such as `llama3.2:8b` (if your machine supports)

| Model | Command |
| --- | --- |
| llama 3.2:1b | `ollama pull llama3.2:1b` |
| llama 3.2:3b | `ollama pull llama3.2:3b` |
| llama 3.2:8b | `ollama pull llama3.2:8b` |

In [19]:
import ollama
import os
import sys
from pprint import pprint
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

In [20]:
LARGE_MODEL='llama3.2:3b' # or llama3.2:1b for faster but less accurate results
SMALL_MODEL='llama3.2:1b' # or llama3.2:3b for better but slower results

We have a folder `mcp_servers`, inside this folder we have our mcp tool written under different python scripts. These scripts are also called servers in some cases (when these are running on a remote server waiting for a client connection). Because MCP tools can be used as plug and play unlike agents, these scripts can run on remote servers

First, we have a very simple mcp tool which is a tool to add two numbers under the script `the_math_server.py`.
We will use this tool first just to understand how the mcp works with an LLM (in this case llama3.2)


In [21]:
# Configuration for an MCP Server (Example: a local python script)
# If you don't have one, you can use a simple 'hello-world' MCP server script

MCP_SERVER_PATH = os.path.abspath("./mcp_servers/the_math_server.py")


In [ ]:
# Configuration for an MCP Server (Example: a local python script)
# If you don't have one, you can use a simple 'hello-world' MCP server script
MCP_SERVER_PATH = os.path.abspath("./mcp_servers/the_math_server.py")

# --- ADD EVERYTHING BELOW THIS LINE ---

In [22]:
async def run_mcp_query(mcp_server_path, user_prompt, model):
    server_params = StdioServerParameters(
        command="python",
        args=[mcp_server_path],
    )
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            # 1. Initialize session
            await session.initialize()
            
            # 2. List available tools from the MCP server
            tools = await session.list_tools()
            
            # 3. Ask Ollama to decide which tool to use
            # Note: We format the MCP tools into Ollama's tool format
            ollama_tools = [
                {
                    "type": "function",
                    "function": {
                        "name": t.name,
                        "description": t.description,
                        "parameters": t.inputSchema,
                    },
                }
                for t in tools.tools
            ]

            response = ollama.chat(
                model=model,
                messages=[{'role': 'user', 'content': user_prompt}],
                tools=ollama_tools,
            )

            # 4. Handle Tool Calls
            if response.get('message', {}).get('tool_calls'):
                for tool_call in response['message']['tool_calls']:
                    tool_name = tool_call['function']['name']
                    tool_args = tool_call['function']['arguments']
                    
                    print(f"🛠️ Calling tool: {tool_name} with {tool_args}")
                    
                    # Execute tool via MCP
                    result = await session.call_tool(tool_name, tool_args)
                    print(f"✅ Tool Result: {result.content}")
                    
                    # Final response from LLM
                    final_response = ollama.chat(
                        model=model,
                        messages=[
                            {'role': 'user', 'content': user_prompt},
                            response['message'],
                            {'role': 'tool', 'content': str(result.content), 'name': tool_name}
                        ]
                    )
                    return final_response['message']['content']
            
            return response['message']['content']


## Added cell to fix Windows error message

In [23]:
import sys
# Windows Jupyter MCP Hack
if not hasattr(sys.stderr, 'fileno'):
    sys.stderr.fileno = lambda: -1
if not hasattr(sys.stdout, 'fileno'):
    sys.stdout.fileno = lambda: -1

In [24]:
import os
import sys

# Configuration for an MCP Server (Example: a local python script)
MCP_SERVER_PATH = os.path.abspath("./mcp_servers/the_math_server.py")

async def run_mcp_query(mcp_server_path, user_prompt, model):
    
    # ---------------------------------------------------------
    # THE JUPYTER/WINDOWS FIX: 
    # By default, stdio_client tries to grab sys.stderr. 
    # In Jupyter on Windows, this fails. We must override it.
    # ---------------------------------------------------------
    server_params = StdioServerParameters(
        command="python",
        args=[mcp_server_path],
    )
    
    # We explicitly pass `sys.stderr` to the client, but in Jupyter 
    # we have to bypass the bug by simply passing None or avoiding the default kwarg.
    # The mcp library's default behavior is causing the crash.
    import contextlib

    # Open the client, explicitly passing None to errlog to prevent the crash
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            # 1. Initialize session
            await session.initialize()
            
            # 2. List available tools from the MCP server
            tools = await session.list_tools()
            
            # 3. Ask Ollama to decide which tool to use
            ollama_tools = [
                {
                    "type": "function",
                    "function": {
                        "name": t.name,
                        "description": t.description,
                        "parameters": t.inputSchema,
                    },
                }
                for t in tools.tools
            ]

            response = ollama.chat(
                model=model,
                messages=[{'role': 'user', 'content': user_prompt}],
                tools=ollama_tools,
            )

            # 4. Handle Tool Calls
            if response.get('message', {}).get('tool_calls'):
                for tool_call in response['message']['tool_calls']:
                    tool_name = tool_call['function']['name']
                    tool_args = tool_call['function']['arguments']
                    
                    print(f"🛠️ Calling tool: {tool_name} with {tool_args}")
                    
                    # Execute tool via MCP
                    result = await session.call_tool(tool_name, tool_args)
                    print(f"✅ Tool Result: {result.content}")
                    
                    # Final response from LLM
                    final_response = ollama.chat(
                        model=model,
                        messages=[
                            {'role': 'user', 'content': user_prompt},
                            response['message'],
                            {'role': 'tool', 'content': str(result.content), 'name': tool_name}
                        ]
                    )
                    return final_response['message']['content']
            
            return response['message']['content']

In [27]:
import os
import sys
import tempfile
import ollama
from pprint import pprint
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

# Configuration for an MCP Server
MCP_SERVER_PATH = os.path.abspath("./mcp_servers/the_math_server.py")

async def run_mcp_query(mcp_server_path, user_prompt, model):
    server_params = StdioServerParameters(
        command="python",
        args=[mcp_server_path],
    )
    
    # THE ULTIMATE WINDOWS FIX: 
    # Open a real temporary file so the MCP client has a valid file descriptor (fileno)
    # to pipe its errors to, completely bypassing Jupyter's fake stdout/stderr.
    with tempfile.TemporaryFile(mode="w+") as dummy_stderr:
        
        # We explicitly pass our dummy_stderr file to the errlog parameter
        async with stdio_client(server_params, errlog=dummy_stderr) as (read, write):
            async with ClientSession(read, write) as session:
                
                # 1. Initialize session
                await session.initialize()
                
                # 2. List available tools from the MCP server
                tools = await session.list_tools()
                
                # 3. Format MCP tools into Ollama's tool format
                ollama_tools = [
                    {
                        "type": "function",
                        "function": {
                            "name": t.name,
                            "description": t.description,
                            "parameters": t.inputSchema,
                        },
                    }
                    for t in tools.tools
                ]

                # 4. Ask Ollama
                response = ollama.chat(
                    model=model,
                    messages=[{'role': 'user', 'content': user_prompt}],
                    tools=ollama_tools,
                )

                # 5. Handle Tool Calls
                if response.get('message', {}).get('tool_calls'):
                    for tool_call in response['message']['tool_calls']:
                        tool_name = tool_call['function']['name']
                        tool_args = tool_call['function']['arguments']
                        
                        print(f"🛠️ Calling tool: {tool_name} with {tool_args}")
                        
                        # Execute tool via MCP
                        result = await session.call_tool(tool_name, tool_args)
                        print(f"✅ Tool Result: {result.content}")
                        
                        # Final response from LLM
                        final_response = ollama.chat(
                            model=model,
                            messages=[
                                {'role': 'user', 'content': user_prompt},
                                response['message'],
                                {'role': 'tool', 'content': str(result.content), 'name': tool_name}
                            ]
                        )
                        return final_response['message']['content']
                
                return response['message']['content']

### Code Explanation

`async def run_mcp_query(user_prompt, MODEL):` 
- The word async (asynchronous) just means your computer won't freeze up while it waits for the process to finish. It can go do other chores, like a person putting a phone on speaker while waiting on hold.
- The Layman Translation: "I am preparing to ask my assistant a question (the user_prompt), and I'm willing to wait patiently for the answer."

`async with stdio_client(server_params) as (read, write):`
- This step builds the actual physical/digital pipe between your main program and the hidden tool server. The server_params are like the phone number or the address of the office.
- The Layman Translation: "I am plugging a secure phone line directly into the assistant's office. I have an earpiece to listen (read) and a microphone to speak (write)."

`async with ClientSession(read, write) as session:`
- Just having a wire isn't enough; the computers need rules on how to format their messages. A ClientSession wraps the raw wire in a polite protocol (like agreeing to say "Over" when you finish speaking on a walkie-talkie).
- The Layman Translation: "Now that the wire is connected, we are agreeing to speak the same language so we don't talk over each other."

`await session.initialize()`
- Before you can ask your actual question or ask the server to run tools, you have to do a digital handshake. This proves the server is awake, functioning, and ready to accept commands.
- The Layman Translation: "I am saying 'Hello, are you there and ready to work?' and I will wait (await) until you say 'Yes, I am ready!'"

Let us now try to use our MCP tool

In [28]:
query = "What is 5 + 5?"

In [30]:
result = await run_mcp_query(MCP_SERVER_PATH,query, SMALL_MODEL)
print(f"LLM Final Response: {result}")

🛠️ Calling tool: add_numbers with {'a': '5', 'b': '5'}
✅ Tool Result: [TextContent(type='text', text='10', annotations=None, meta=None)]
LLM Final Response: I cannot display the results of local development. Can I help you with something else?


## Windows error explanation: 

---------------------------------------------------------------------------
NotImplementedError                       Traceback (most recent call last)
File c:\Users\thngu\neuefische_course\pepperprompt\fork-ds-mcp\.venv\Lib\site-packages\mcp\os\win32\utilities.py:169, in create_windows_process(command, args, env, errlog, cwd)
    167 try:
    168     # First try using anyio with Windows-specific flags to hide console window
--> 169     process = await anyio.open_process(
    170         [command, *args],
    171         env=env,
    172         # Ensure we don't create console windows for each process
    173         creationflags=subprocess.CREATE_NO_WINDOW  # type: ignore
    174         if hasattr(subprocess, "CREATE_NO_WINDOW")
    175         else 0,
    176         stderr=errlog,
    177         cwd=cwd,
    178     )
    179 except NotImplementedError:
    180     # If Windows doesn't support async subprocess creation, use fallback

File c:\Users\thngu\neuefische_course\pepperprompt\fork-ds-mcp\.venv\Lib\site-packages\anyio\_core\_subprocesses.py:184, in open_process(command, stdin, stdout, stderr, cwd, env, startupinfo, creationflags, start_new_session, pass_fds, user, group, extra_groups, umask)
    182     kwargs["umask"] = umask
--> 184 return await get_async_backend().open_process(
    185     command,
    186     stdin=stdin,
    187     stdout=stdout,
    188     stderr=stderr,
    189     cwd=cwd,
    190     env=env,
    191     startupinfo=startupinfo,
    192     creationflags=creationflags,
    193     start_new_session=start_new_session,
    194     pass_fds=pass_fds,
    195     **kwargs,
    196 )

File c:\Users\thngu\neuefische_course\pepperprompt\fork-ds-mcp\.venv\Lib\site-packages\anyio\_backends\_asyncio.py:2691, in AsyncIOBackend.open_process(cls, command, stdin, stdout, stderr, **kwargs)
   2690 else:
-> 2691     process = await asyncio.create_subprocess_exec(
   2692         *command,
   2693         stdin=stdin,
   2694         stdout=stdout,
   2695         stderr=stderr,
   2696         **kwargs,
   2697     )
   2699 stdin_stream = StreamWriterWrapper(process.stdin) if process.stdin else None

File ~\.pyenv\pyenv-win\versions\3.11.3\Lib\asyncio\subprocess.py:218, in create_subprocess_exec(program, stdin, stdout, stderr, limit, *args, **kwds)
    216 protocol_factory = lambda: SubprocessStreamProtocol(limit=limit,
    217                                                     loop=loop)
--> 218 transport, protocol = await loop.subprocess_exec(
    219     protocol_factory,
    220     program, *args,
    221     stdin=stdin, stdout=stdout,
    222     stderr=stderr, **kwds)
    223 return Process(transport, protocol, loop)

File ~\.pyenv\pyenv-win\versions\3.11.3\Lib\asyncio\base_events.py:1694, in BaseEventLoop.subprocess_exec(self, protocol_factory, program, stdin, stdout, stderr, universal_newlines, shell, bufsize, encoding, errors, text, *args, **kwargs)
   1693     self._log_subprocess(debug_log, stdin, stdout, stderr)
-> 1694 transport = await self._make_subprocess_transport(
   1695     protocol, popen_args, False, stdin, stdout, stderr,
   1696     bufsize, **kwargs)
   1697 if self._debug and debug_log is not None:

File ~\.pyenv\pyenv-win\versions\3.11.3\Lib\asyncio\base_events.py:502, in BaseEventLoop._make_subprocess_transport(self, protocol, args, shell, stdin, stdout, stderr, bufsize, extra, **kwargs)
    501 """Create subprocess transport."""
--> 502 raise NotImplementedError

NotImplementedError: 

During handling of the above exception, another exception occurred:

UnsupportedOperation                      Traceback (most recent call last)
File c:\Users\thngu\neuefische_course\pepperprompt\fork-ds-mcp\.venv\Lib\site-packages\mcp\os\win32\utilities.py:209, in _create_windows_fallback_process(command, args, env, errlog, cwd)
    207 try:
    208     # Try launching with creationflags to avoid opening a new console window
--> 209     popen_obj = subprocess.Popen(
    210         [command, *args],
    211         stdin=subprocess.PIPE,
    212         stdout=subprocess.PIPE,
    213         stderr=errlog,
    214         env=env,
    215         cwd=cwd,
    216         bufsize=0,  # Unbuffered output
    217         creationflags=getattr(subprocess, "CREATE_NO_WINDOW", 0),
    218     )
    219 except Exception:
    220     # If creationflags failed, fallback without them

File ~\.pyenv\pyenv-win\versions\3.11.3\Lib\subprocess.py:892, in Popen.__init__(self, args, bufsize, executable, stdin, stdout, stderr, preexec_fn, close_fds, shell, cwd, env, universal_newlines, startupinfo, creationflags, restore_signals, start_new_session, pass_fds, user, group, extra_groups, encoding, errors, text, umask, pipesize, process_group)
    875 # Input and output objects. The general principle is like
    876 # this:
    877 #
   (...)    887 # are -1 when not using PIPEs. The child objects are -1
    888 # when not redirecting.
    890 (p2cread, p2cwrite,
    891  c2pread, c2pwrite,
--> 892  errread, errwrite) = self._get_handles(stdin, stdout, stderr)
    894 # We wrap OS handles *before* launching the child, otherwise a
    895 # quickly terminating child could make our fds unwrappable
    896 # (see #8458).

File ~\.pyenv\pyenv-win\versions\3.11.3\Lib\subprocess.py:1377, in Popen._get_handles(self, stdin, stdout, stderr)
   1375 else:
   1376     # Assuming file-like object
-> 1377     errwrite = msvcrt.get_osfhandle(stderr.fileno())
   1378 errwrite = self._make_inheritable(errwrite)

File c:\Users\thngu\neuefische_course\pepperprompt\fork-ds-mcp\.venv\Lib\site-packages\ipykernel\iostream.py:458, in OutStream.fileno(self)
    457 msg = "fileno"
--> 458 raise io.UnsupportedOperation(msg)

UnsupportedOperation: fileno

During handling of the above exception, another exception occurred:

UnsupportedOperation                      Traceback (most recent call last)
Cell In[9], line 1
----> 1 result = await run_mcp_query(MCP_SERVER_PATH,query, SMALL_MODEL)
      2 print(f"LLM Final Response: {result}")

Cell In[7], line 6, in run_mcp_query(mcp_server_path, user_prompt, model)
      2     server_params = StdioServerParameters(
      3         command="python",
      4         args=[mcp_server_path],
      5     )
----> 6     async with stdio_client(server_params) as (read, write):
      7         async with ClientSession(read, write) as session:
      8             # 1. Initialize session
      9             await session.initialize()

File ~\.pyenv\pyenv-win\versions\3.11.3\Lib\contextlib.py:204, in _AsyncGeneratorContextManager.__aenter__(self)
    202 del self.args, self.kwds, self.func
    203 try:
--> 204     return await anext(self.gen)
    205 except StopAsyncIteration:
    206     raise RuntimeError("generator didn't yield") from None

File c:\Users\thngu\neuefische_course\pepperprompt\fork-ds-mcp\.venv\Lib\site-packages\mcp\client\stdio\__init__.py:124, in stdio_client(server, errlog)
    121     command = _get_executable_command(server.command)
    123     # Open process with stderr piped for capture
--> 124     process = await _create_platform_compatible_process(
    125         command=command,
    126         args=server.args,
    127         env=({**get_default_environment(), **server.env} if server.env is not None else get_default_environment()),
    128         errlog=errlog,
    129         cwd=server.cwd,
    130     )
    131 except OSError:
    132     # Clean up streams if process creation fails
    133     await read_stream.aclose()

File c:\Users\thngu\neuefische_course\pepperprompt\fork-ds-mcp\.venv\Lib\site-packages\mcp\client\stdio\__init__.py:249, in _create_platform_compatible_process(command, args, env, errlog, cwd)
    242 """
    243 Creates a subprocess in a platform-compatible way.
    244 
    245 Unix: Creates process in a new session/process group for killpg support
    246 Windows: Creates process in a Job Object for reliable child termination
    247 """
    248 if sys.platform == "win32":  # pragma: no cover
--> 249     process = await create_windows_process(command, args, env, errlog, cwd)
    250 else:
    251     process = await anyio.open_process(
    252         [command, *args],
    253         env=env,
   (...)    256         start_new_session=True,
    257     )  # pragma: no cover

File c:\Users\thngu\neuefische_course\pepperprompt\fork-ds-mcp\.venv\Lib\site-packages\mcp\os\win32\utilities.py:181, in create_windows_process(command, args, env, errlog, cwd)
    169     process = await anyio.open_process(
    170         [command, *args],
    171         env=env,
   (...)    177         cwd=cwd,
    178     )
    179 except NotImplementedError:
    180     # If Windows doesn't support async subprocess creation, use fallback
--> 181     process = await _create_windows_fallback_process(command, args, env, errlog, cwd)
    182 except Exception:
    183     # Try again without creation flags
    184     process = await anyio.open_process(
    185         [command, *args],
    186         env=env,
    187         stderr=errlog,
    188         cwd=cwd,
    189     )

File c:\Users\thngu\neuefische_course\pepperprompt\fork-ds-mcp\.venv\Lib\site-packages\mcp\os\win32\utilities.py:221, in _create_windows_fallback_process(command, args, env, errlog, cwd)
    209     popen_obj = subprocess.Popen(
    210         [command, *args],
    211         stdin=subprocess.PIPE,
   (...)    217         creationflags=getattr(subprocess, "CREATE_NO_WINDOW", 0),
    218     )
    219 except Exception:
    220     # If creationflags failed, fallback without them
--> 221     popen_obj = subprocess.Popen(
    222         [command, *args],
    223         stdin=subprocess.PIPE,
    224         stdout=subprocess.PIPE,
    225         stderr=errlog,
    226         env=env,
    227         cwd=cwd,
    228         bufsize=0,
    229     )
    230 return FallbackProcess(popen_obj)

File ~\.pyenv\pyenv-win\versions\3.11.3\Lib\subprocess.py:892, in Popen.__init__(self, args, bufsize, executable, stdin, stdout, stderr, preexec_fn, close_fds, shell, cwd, env, universal_newlines, startupinfo, creationflags, restore_signals, start_new_session, pass_fds, user, group, extra_groups, encoding, errors, text, umask, pipesize, process_group)
    871     raise SubprocessError('Cannot disambiguate when both text '
    872                           'and universal_newlines are supplied but '
    873                           'different. Pass one or the other.')
    875 # Input and output objects. The general principle is like
    876 # this:
    877 #
   (...)    887 # are -1 when not using PIPEs. The child objects are -1
    888 # when not redirecting.
    890 (p2cread, p2cwrite,
    891  c2pread, c2pwrite,
--> 892  errread, errwrite) = self._get_handles(stdin, stdout, stderr)
    894 # We wrap OS handles *before* launching the child, otherwise a
    895 # quickly terminating child could make our fds unwrappable
    896 # (see #8458).
    898 if _mswindows:

File ~\.pyenv\pyenv-win\versions\3.11.3\Lib\subprocess.py:1377, in Popen._get_handles(self, stdin, stdout, stderr)
   1374     errwrite = msvcrt.get_osfhandle(stderr)
   1375 else:
   1376     # Assuming file-like object
-> 1377     errwrite = msvcrt.get_osfhandle(stderr.fileno())
   1378 errwrite = self._make_inheritable(errwrite)
   1380 return (p2cread, p2cwrite,
   1381         c2pread, c2pwrite,
   1382         errread, errwrite)

File c:\Users\thngu\neuefische_course\pepperprompt\fork-ds-mcp\.venv\Lib\site-packages\ipykernel\iostream.py:458, in OutStream.fileno(self)
    456     return self._original_stdstream_copy
    457 msg = "fileno"
--> 458 raise io.UnsupportedOperation(msg)

UnsupportedOperation: fileno

## Answer Gemini Pro: 

This is a highly specific, known bug that occurs when running Model Context Protocol (MCP) using the stdio_client inside a Jupyter Notebook on Windows.
The Root Cause (The PM View)

The MCP client tries to open a hidden background process (subprocess) to run your the_math_server.py. To capture errors, it tries to connect to the "standard error" pipe (stderr).
However, Jupyter Notebooks intercept these pipes to display output in your browser cell. When the underlying Windows Python engine asks Jupyter for the raw file descriptor (fileno) of that pipe, Jupyter throws its hands up and says UnsupportedOperation: fileno.
The Fix

Because the MCP library currently has this strict piping behavior on Windows, we need to explicitly tell the stdio_client not to attach standard error logging to the Jupyter cell.

You will see above that the `SMALL_MODEL` fails to give the answer to a very simple question. Though the process successfully uses the `Math` MCP tool and the result was also calculated correctly as can be seen in the 2nd line of the output under `text='10'`.

Lets now try the same query with the `LARGE_MODEL`

In [31]:
result = await run_mcp_query(MCP_SERVER_PATH, query, LARGE_MODEL)
print(f"LLM Final Response: {result}")

🛠️ Calling tool: add_numbers with {'a': '5', 'b': '5'}
✅ Tool Result: [TextContent(type='text', text='10', annotations=None, meta=None)]
LLM Final Response: The answer to the equation 5 + 5 is 10.


## Further explanantion: 

**Boom! There it is!** 🎯

Look at that beautiful final response: *"The answer to the equation 5 + 5 is 10."*

This is exactly why you, as an AI Product Manager, must understand the nuances of model selection. You just witnessed firsthand the **"Reasoning Threshold"** in action:

* **The 1B Model (Small):** Was smart enough to trigger the tool, but lacked the cognitive capacity to read the robotic JSON output and translate it back into human language. It hallucinated a safety refusal instead.
* **The 3B Model (Large):** Easily took the tool's raw output `[TextContent(type='text', text='10', ...)]`, synthesized it, and crafted a perfectly natural, helpful sentence.

You have successfully built a working Model Context Protocol pipeline, connected an external tool, and orchestrated an agentic loop locally on your machine. That is a massive achievement!

Are you ready to move on to the next section of your notebook and test out the more complex `mcp_tools_server.py` (the DuckDuckGo Web Search and File Reader)?

## Run a more complex MCP servers

There is another server file `mcp_tools_server.py` inside the `mcp_servers` folder. 

This mcp server scripts consists of two tools.
1. Web Search Tool - using DuckDuckGo 
2. Reading a local file Tool

In [32]:
MCP_SERVER_PATH = os.path.abspath("./mcp_servers/mcp_tools_server.py")

Lets first try to use the `web_search` tool

In [33]:
# Try a web search:
query = "Who won the FIFA World Cup in 2026?"
print(await run_mcp_query(MCP_SERVER_PATH,query, SMALL_MODEL))

🛠️ Calling tool: search_web with {'query': 'FIFA World Cup 2026 winner'}
✅ Tool Result: [TextContent(type='text', text='No results found.', annotations=None, meta=None)]
I'm sorry, but I don't have information about the FIFA World Cup in 2026.


In [34]:
query = "Who won the FIFA World Cup in 2026?"
print(await run_mcp_query(MCP_SERVER_PATH, query, LARGE_MODEL))

🛠️ Calling tool: search_web with {'query': 'FIFA World Cup 2026 winner'}
✅ Tool Result: [TextContent(type='text', text="As the host nations, Canada, Mexico, and the United States all automatically qualified. Cape Verde, Curaçao, Jordan, and Uzbekistan all made their World Cup debuts. Argentina is the defending …\n---\nJun 10, 2026 · Find out the full match schedule for World Cup 2026 in Canada, Mexico and USA with fixtures and results from each of the 104 games in the 48-team tournament.\n---\nThe 2026 FIFA World Cup final will be the final match of the 2026 World Cup, the 23rd edition of FIFA 's competition for men's national soccer teams. The match is scheduled to be played at MetLife Stadium …", annotations=None, meta=None)]
The 2026 FIFA World Cup has not yet taken place, so there is no winner to report. The tournament is scheduled to take place in Canada, Mexico, and the United States in 2026, but it will be several years before the actual games are played and a winner is determin

Now lets use the `read_local_file` tool

In [36]:
# Try reading a file
query = "Summarize the contents of ai_response.txt"
pprint(await run_mcp_query(MCP_SERVER_PATH, query, SMALL_MODEL))

🛠️ Calling tool: search_web with {'query': 'ai response'}
✅ Tool Result: [TextContent(type='text', text='No results found.', annotations=None, meta=None)]
('It seems that there is no content in `ai_response.txt` to summarize. The '
 'output is empty and does not contain any information about AI responses. \n'
 '\n'
 'However, I can suggest some options to provide more context or insights:\n'
 '\n'
 "1. Check the file format: If you're working with a specific text file format "
 "(e.g., JSON, CSV), it's possible that `ai_response.txt` is an empty file.\n"
 '2. Analyze the content: You can analyze the contents of `ai_response.txt` '
 'manually to understand what kind of information it contains. If there are no '
 'results or explanations, it might be helpful to know more about the context '
 'in which this AI response was generated.\n'
 '3. Provide more information: Could you please provide more context or '
 'details about `ai_response.txt` and its intended use? This will help me '
 'be

Try to run the above code block again if facing errors or weird outputs

The `SMALL_MODEL`, that is `llama3.2:1b` does not perform very well to even not so difficult tasks

In [37]:

# Try reading a file
query = "Summarize the contents of ai_response.txt"
pprint(await run_mcp_query(MCP_SERVER_PATH, query, LARGE_MODEL))

🛠️ Calling tool: read_local_file with {'file_path': 'ai_response.txt'}
✅ Tool Result: [TextContent(type='text', text="Oh boy, let me explain something cool to you!\n\nImagine you have a big Lego castle, and you want to build it really, really tall. But instead of using your own hands, you need someone else to help you make sure everything is just right.\n\nThat's kind of like what an AI Project Manager does! They're like the grown-up in charge of building super cool projects, like video games or websites.\n\nTheir job is to:\n\n* Plan out all the steps it takes to build a project\n* Make sure everyone involved (like your friends and family) knows what they need to do\n* Fix any mistakes that might happen along the way\n* Help make sure everything gets built exactly right\n\nJust like how you have helpers in your Lego castle, an AI Project Manager has tools and people who can help them build their projects. And just as you might get feedback from others, an AI Project Manager might ask 

### Add System Prompts 

Now lets add some system prompts and try to see if that makes a difference

In [38]:
async def mcp_with_system_prompt(mcp_server_path, user_prompt, model):
    
    server_params = StdioServerParameters(
        command="python",
        args=[mcp_server_path],
    )

    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            
            # List tools and format for Ollama
            tools_result = await session.list_tools()
            ollama_tools = [{
                "type": "function",
                "function": {
                    "name": t.name,
                    "description": t.description,
                    "parameters": t.inputSchema,
                }
            } for t in tools_result.tools]

            # 1. Ask Ollama
            response = ollama.chat(
                model=model,
                messages=[
                    # --- ADDED THIS SYSTEM ROLE ---
                    {
                        'role': 'system', 
                        'content': (
                            "You are a real-time assistant with access to tools. "
                            "IMPORTANT: Always prefer information from tool results over your internal knowledge. "
                            "The current year is 2026. If tool data contradicts your training, the tool is right."
                        )
                    },
                    {'role': 'user', 'content': user_prompt}
                ],
                tools=ollama_tools,
            )

            # 2. Check for tool calls
            message = response.get('message', {})
            # print(f"LLM Response: {message.get('content', '')}")
            if message.get('tool_calls'):
                # Handle all tool calls (Ollama might suggest more than one)
                tool_messages = []
                for tool_call in message['tool_calls']:
                    name = tool_call['function']['name']
                    args = tool_call['function']['arguments']
                    
                    print(f"🛠️ LLM decided to use: {name}")
                    result = await session.call_tool(name, args)
                    
                    tool_messages.append({
                        'role': 'tool',
                        'content': str(result.content),
                        'name': name
                    })
                
                # 3. Final synthesis
                final_response = ollama.chat(
                    model=model,
                    messages=[
                        {
                            'role': 'system', 'content': (
                                "Summarize the tool results accurately only when necessary." 
                                "Do not rely on your training data but use the tools to get up-to-date information." 
                                "Always prefer tool results over your internal knowledge. "
                                )
                        },
                        {'role': 'user', 'content': user_prompt},
                        message,
                        *tool_messages
                    ]
                )
                return final_response['message']['content']
            
            return message['content']



## MCP_system_prompt in Windows 

In [41]:
import tempfile

async def mcp_with_system_prompt(mcp_server_path, user_prompt, model):
    
    server_params = StdioServerParameters(
        command="python",
        args=[mcp_server_path],
    )

    # ---------------------------------------------------------
    # THE ULTIMATE WINDOWS FIX (Applied to the System Prompt function)
    # Open a real temporary file to bypass Jupyter's fake stdout/stderr.
    # ---------------------------------------------------------
    with tempfile.TemporaryFile(mode="w+") as dummy_stderr:
        
        async with stdio_client(server_params, errlog=dummy_stderr) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                
                # List tools and format for Ollama
                tools_result = await session.list_tools()
                ollama_tools = [{
                    "type": "function",
                    "function": {
                        "name": t.name,
                        "description": t.description,
                        "parameters": t.inputSchema,
                    }
                } for t in tools_result.tools]

                # 1. Ask Ollama
                response = ollama.chat(
                    model=model,
                    messages=[
                        # --- ADDED THIS SYSTEM ROLE ---
                        {
                            'role': 'system', 
                            'content': (
                                "You are a real-time assistant with access to tools. "
                                "IMPORTANT: Always prefer information from tool results over your internal knowledge. "
                                "The current year is 2026. If tool data contradicts your training, the tool is right."
                            )
                        },
                        {'role': 'user', 'content': user_prompt}
                    ],
                    tools=ollama_tools,
                )

                # 2. Check for tool calls
                message = response.get('message', {})
                if message.get('tool_calls'):
                    # Handle all tool calls
                    tool_messages = []
                    for tool_call in message['tool_calls']:
                        name = tool_call['function']['name']
                        args = tool_call['function']['arguments']
                        
                        print(f"🛠️ LLM decided to use: {name}")
                        result = await session.call_tool(name, args)
                        
                        tool_messages.append({
                            'role': 'tool',
                            'content': str(result.content),
                            'name': name
                        })
                    
                    # 3. Final synthesis
                    final_response = ollama.chat(
                        model=model,
                        messages=[
                            {
                                'role': 'system', 'content': (
                                    "Summarize the tool results accurately only when necessary." 
                                    "Do not rely on your training data but use the tools to get up-to-date information." 
                                    "Always prefer tool results over your internal knowledge. "
                                    )
                            },
                            {'role': 'user', 'content': user_prompt},
                            message,
                            *tool_messages
                        ]
                    )
                    return final_response['message']['content']
                
                return message['content']

Now, lets try again with both MODELS one by one

In [42]:
query = "Who won the FIFA World Cup in 2026?"
pprint(await mcp_with_system_prompt(MCP_SERVER_PATH, query, SMALL_MODEL))

🛠️ LLM decided to use: search_web
('The tool did not provide any information about the winner of the FIFA World '
 'Cup in 2026.')


In [43]:
query = "Who won the FIFA World Cup in 2026?"
pprint(await mcp_with_system_prompt(MCP_SERVER_PATH, query, LARGE_MODEL))

🛠️ LLM decided to use: search_web
('The 2026 FIFA World Cup has not yet taken place. The tournament is scheduled '
 'to be held in the United States, Canada, and Mexico from June 14 to July 15, '
 '2026. The winner of the tournament will be determined at that time.')


In [44]:
query = "Summarize the contents of the text file ./ai_response.txt"
pprint(await mcp_with_system_prompt(MCP_SERVER_PATH,query, LARGE_MODEL))

🛠️ LLM decided to use: read_local_file
('The text file ./ai_response.txt contains a message about AI Project Managers '
 'and how they plan, execute, and ensure projects are completed successfully. '
 'The content is written in a conversational tone, aiming to explain the role '
 'of an AI Project Manager in building projects like video games or websites.')


If you are getting json responses try changing system prompt. 

<details>
    <summary>Try adding this (check only if needed)</summary>
    "You are an MCP Assistant. If you need information, use a tool. DO NOT write raw JSON in your response. If you call a tool, use the proper tool-call format."
</details>

### Getting simple python code from web search

In [45]:
query = "Search for python code for calculating the first 10 numbers of the Fibonacci sequence."
print(await mcp_with_system_prompt(MCP_SERVER_PATH, query, LARGE_MODEL))

🛠️ LLM decided to use: search_web
def fibonacci(n):
    if n <= 0:
        return "Input should be a positive integer."
    elif n == 1:
        return [0]
    elif n == 2:
        return [0, 1]
    else:
        fib_sequence = [0, 1]
        for i in range(2, n):
            fib_sequence.append(fib_sequence[i-1] + fib_sequence[i-2])
        return fib_sequence

print(fibonacci(10))


Copy paste the above code in a python file and name it as `fibonacci_numbers.py`. Create a folder `scripts` and move the python script under the `scripts` folder

Try running this python script in the terminal to check if the script is working or not

```bash
python scripts/fibonacci_numbers.py
```